# Adaptive RAG
### Routing each query to the cheapest model that can answer it correctly

Corpus: `NIST AI RMF (AI.100-1)` + `OWASP Top 10 for LLMs (2025)` merged into one index — a mix of simple lookups ("what does X mean") and analytical questions that need synthesis across both documents.

## Step 1: Build the pipeline

In [1]:
!pip install langchain langchain-community langchain-openai langchain-ollama langchain-text-splitters faiss-cpu pypdf python-dotenv -q


[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
from dotenv import load_dotenv
load_dotenv()

from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_ollama import ChatOllama

docs = []
for path, source in [("NIST.AI.100-1.pdf", "NIST AI RMF"), ("OWASP-Top-10-for-LLMs-v2025.pdf", "OWASP LLM Top 10")]:
    pages = PyPDFLoader(path).load()
    for p in pages:
        p.metadata["source"] = source
    docs.extend(pages)

splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
chunks = splitter.split_documents(docs)

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
vector_store = FAISS.from_documents(chunks, embeddings)

print(f"Loaded {len(docs)} pages -> {len(chunks)} chunks -> {vector_store.index.ntotal} vectors")

C:\Users\shiva\AppData\Local\Temp\ipykernel_23024\1203396366.py:4: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


C:\Users\shiva\.pyenv\pyenv-win\versions\3.12.10\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Error -3 while decompressing data: invalid code lengths set


Error -3 while decompressing data: invalid code lengths set


Error -3 while decompressing data: invalid code lengths set


Error -3 while decompressing data: invalid code lengths set


Loaded 93 pages -> 281 chunks -> 281 vectors


## Step 2: Two generation tiers
Adaptive RAG's only decision here is *which model answers* — retrieval is identical for every query. `llama3.2:3b` is fast and free but weaker at synthesis; `gpt-4o-mini` is slower and costs more but handles multi-part, analytical questions far better.

In [3]:
simple_llm = ChatOllama(model="llama3.2:3b", temperature=0)
complex_llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
TIERS = {"SIMPLE": ("llama3.2:3b", simple_llm), "COMPLEX": ("gpt-4o-mini", complex_llm)}

## Step 3: Classifier — the local model routes each query
A single cheap call to the local model decides SIMPLE vs COMPLEX before any expensive generation happens. Keeping the classifier local means routing itself costs nothing.

In [4]:
CLASSIFY_PROMPT = """Classify the question below as exactly one word: SIMPLE or COMPLEX.

SIMPLE = asks for one fact, definition, or single item that a short passage directly states.
COMPLEX = requires comparing, connecting, or synthesizing information across multiple sections or documents; multi-part or analytical questions.

Question: {query}
Answer with exactly one word:"""

def classify(query):
    label = simple_llm.invoke(CLASSIFY_PROMPT.format(query=query)).content.strip().upper()
    return "COMPLEX" if "COMPLEX" in label else "SIMPLE"

## Step 4: Router — classify, retrieve, generate with the selected tier
Retrieval is untouched by the routing decision; only the generation model changes.

In [5]:
RAG_PROMPT = """Answer the question using only the following context. Be concise.

Context:
{context}

Question: {query}
Answer:"""

def answer(query, k=4):
    tier = classify(query)
    model_name, llm = TIERS[tier]
    retrieved = vector_store.similarity_search(query, k=k)
    context = "\n\n".join(c.page_content for c in retrieved)
    response = llm.invoke(RAG_PROMPT.format(context=context, query=query)).content.strip()
    return tier, model_name, response

## Step 5: Try it on a range of queries
Two single-fact lookups and two questions that force synthesis across both documents.

In [6]:
queries = [
    "According to OWASP, what is Prompt Injection?",
    "What are the four core functions of the NIST AI Risk Management Framework?",
    "How does the NIST AI RMF's MANAGE function relate to the risk OWASP describes under Unbounded Consumption?",
    "Compare how NIST and OWASP each address the risk of a model leaking sensitive data, and say which framework gives more concrete mitigations.",
]

for q in queries:
    tier, model_name, ans = answer(q)
    print(f"[{tier:7s} -> {model_name}] {q}")
    print(f"  {ans}\n")

[SIMPLE  -> llama3.2:3b] According to OWASP, what is Prompt Injection?
  Prompt Injection occurs when user prompts alter the LLM's behavior or output in unintended ways, potentially causing the model to violate guidelines, generate harmful content, enable unauthorized access, or influence critical decisions.



[SIMPLE  -> llama3.2:3b] What are the four core functions of the NIST AI Risk Management Framework?
  The four core functions of the NIST AI Risk Management Framework (AI RMF) are:

1. GOVERN
2. MAP
3. MEASURE
4. MANAGE



[COMPLEX -> gpt-4o-mini] How does the NIST AI RMF's MANAGE function relate to the risk OWASP describes under Unbounded Consumption?
  The NIST AI RMF's MANAGE function addresses AI risks by prioritizing, responding to, and managing them based on assessments from the MAP and MEASURE functions. This relates to the risk of Unbounded Consumption described by OWASP, as it involves identifying and mitigating risks associated with excessive resource usage in AI systems. By implementing risk treatment strategies and monitoring plans, the MANAGE function helps ensure that AI systems do not lead to uncontrolled consumption of resources, thereby enhancing overall risk management.



[COMPLEX -> gpt-4o-mini] Compare how NIST and OWASP each address the risk of a model leaking sensitive data, and say which framework gives more concrete mitigations.
  NIST focuses on comprehensive risk management and security controls, emphasizing the importance of vetting data sources, understanding privacy policies, and applying general security best practices. In contrast, OWASP provides specific guidelines tailored to machine learning applications, including detailed strategies for data sanitization, access controls, and input validation.

OWASP offers more concrete mitigations, such as the implementation of data sanitization techniques and strict access controls, which are directly applicable to preventing sensitive data leakage in LLMs.



## Try it yourself
1. Add a query that's genuinely ambiguous between tiers and see which way the classifier leans.
2. Swap the classifier to `gpt-4o-mini` and compare routing decisions — does a stronger classifier change any verdicts?
3. Add a token counter and sum estimated cost per tier across 20 queries to see the savings adaptive routing gives vs. always using `gpt-4o-mini`.